# Pathway 1: Record and Segment

This notebook guides you through recording new voice data and segmenting it into clips suitable for a TTS training dataset.

**Before you begin:**
- Complete the [Community Data Agreement](../docs/community_agreement_template.md) with each speaker.
- Work through [docs/what_not_to_digitize.md](../docs/what_not_to_digitize.md) to identify any material that should not be recorded.
- Pre-fill `metadata/metadata_template.csv` with speaker consent tiers and cultural protocol notes before the session.


## 1. Recording Requirements

### Equipment
- Omni-directional or cardioid head-mounted microphone
- Quiet room (background noise below -40 dBFS)
- Sample rate: **22050 Hz or higher**, mono, 16-bit PCM

### Recording software
We recommend **Audacity** (free, cross-platform): https://www.audacityteam.org/

Adobe Audition is also supported but is not free. The steps below cover Audacity.

### Audacity settings
- `Edit` → `Preferences` → `Quality`: set Default Sample Rate to **22050 Hz**, Sample Format to **16-bit**
- `Edit` → `Preferences` → `Devices`: set Channels to **1 (Mono)**


## 2. Preparing a Text Corpus

Before recording, prepare a list of sentences for speakers to read. Guidelines:
- Each sentence should be 3–10 seconds when spoken at a natural pace
- Use a variety of phonemes and sentence lengths
- Include sentences meaningful to the community — generic corpora produce generic models
- For language preservation: prioritize sentences that demonstrate vocabulary, grammar, or cultural context the community wants preserved

Format: plain text, one sentence per line. The metadata will be added separately.

## 3. Segmenting in Audacity

If you recorded a long session (rather than individual sentences), use Audacity's Sound Finder to segment it:

1. Open your recording in Audacity
2. Select `Analyze` → `Sound Finder`
3. Adjust `Treat audio below this level as silence` (dB) and `Minimum duration of silence between sounds` until clips are 3–10 seconds
4. Review and adjust label boundaries manually in the label track
5. Export labels: `File` → `Export` → `Export Labels` → save as `Label Track.txt`
6. Export WAVs: `File` → `Export` → `Export Multiple...`
   - Format: WAV, Signed 16-bit PCM
   - Split files based on: **Labels**
   - Name files using: **Label/Track Name**
   - Output folder: `test_data/wavs_export_audacity/`


## 4. Programmatic Segmentation (Alternative)

If you prefer to segment without Audacity, use `scripts/segment_on_silence.py`:

In [ ]:
# Adjust thresholds to match your recording environment
# Use test_en.wav for English
!python ../scripts/segment_on_silence.py \
    --input ../test_data/test_en.wav \
    --output-dir ../test_data/wavs_export_audacity \
    --min-silence-len 700 \
    --silence-thresh -40

## 5. Check Segment Lengths

In [ ]:
import os
import soundfile as sf
import pandas as pd

wavs_dir = "../test_data/wavs_export_audacity/"
rows = []
for fname in sorted(os.listdir(wavs_dir)):
    if fname.endswith(".wav"):
        f, sr = sf.read(os.path.join(wavs_dir, fname))
        duration = len(f) / sr
        rows.append({"file": fname, "duration_s": round(duration, 2)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nTotal: {len(df)} files, {df['duration_s'].sum():.1f}s")
print(f"Outside 3-10s range: {len(df[(df['duration_s'] < 3) | (df['duration_s'] > 10)])} files")

## 6. Fill in Metadata

Open `metadata/metadata_template.csv` and fill in each row:
- `file_id`: the filename without `.wav`
- `speaker_id`: anonymized or named per speaker's preference
- `consent_tier`: as agreed in the Community Data Agreement
- `cultural_protocol`: any restrictions or context
- `knowledge_keeper_reviewed`: set to `true` after community review

Leave `transcript` blank for now — Pathway 2 covers transcription.

See [docs/metadata_schema.md](../docs/metadata_schema.md) for full field documentation.

## Next Steps — Choose Your Transcription Path

| Your language | Notebook |\n|---|---|\n| English, Spanish, Māori, or other Whisper-supported language | [02a_transcribe_whisper.ipynb](02a_transcribe_whisper.ipynb) |\n| Navajo or other language covered by Meta MMS | [02b_transcribe_mms.ipynb](02b_transcribe_mms.ipynb) |\n| Language not covered by either tool, or community prefers manual | [02c_transcribe_manual.ipynb](02c_transcribe_manual.ipynb) |

Not sure? Check the [Whisper language list](https://github.com/openai/whisper#available-models-and-languages) and [MMS language list](https://dl.fbaipublicfiles.com/mms/misc/language_coverage_mms.html).

Also run: [03_snr_quality_analysis.ipynb](03_snr_quality_analysis.ipynb) to check audio quality before finalizing.